# 🚀 Elastic 9.x Vector Search: 사전 데이터 적재 (Setup)
본 노트북은 데모 시연을 위한 한국어 오픈 데이터셋(KLUE) 다운로드, 전처리 및 Elastic 클러스터 고속 색인을 수행합니다.
안전한 시연을 위해 SSL 경고를 숨기고 사용자 인증을 수행합니다.

In [ ]:
import os
import time
import random
import getpass
import warnings
import urllib3
from collections import deque
from datasets import load_dataset
from elasticsearch import Elasticsearch
from elasticsearch.exceptions import ElasticsearchWarning
from elasticsearch.helpers import parallel_bulk

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.simplefilter('ignore', ElasticsearchWarning)

print("🔌 Elasticsearch 클러스터 연결")
es_url = os.environ.get("ES_URL") or input("ES URL [기본값: https://localhost:9200]: ") or "https://localhost:9200"
es_user = os.environ.get("ES_USER") or input("Username [기본값: elastic]: ") or "elastic"
es_pass = os.environ.get("ES_PASSWORD") or getpass.getpass("Password: ")

es = Elasticsearch(es_url, basic_auth=(es_user, es_pass), verify_certs=False)
if es.ping():
    print("✅ 클러스터 연결 성공!")
else:
    print("❌ 연결 실패")

INDEX_REAL_LEGACY = "demo-real-legacy-8x"
INDEX_REAL_MODERN = "demo-real-modern-9x"
VECTOR_DIMS = 1024

## 1. Elastic Inference Service (EIS) 세팅
외부 API 호출 없이 클러스터 내부에서 안전하게 동작하는 Jina v5 (임베딩 및 리랭커) 모델을 활성화합니다.

In [ ]:
print("🧠 EIS 모델 세팅 중 (Jina v5 Inference)...")
try:
    es.inference.put(task_type="text_embedding", inference_id="my-jina-embedding", body={"service": "elastic", "service_settings": {"model_id": "jina-embeddings-v5-text-small"}})
    es.inference.put(task_type="rerank", inference_id="my-jina-reranker", body={"service": "elastic", "service_settings": {"model_id": "jina-reranker-v3"}})
    print("✅ 모델 세팅 완료")
except Exception: 
    print("✅ 모델이 이미 존재합니다.")

## 2. A/B 테스트용 인덱스 아키텍처 구성
* **Legacy (8.x)**: 무거운 `float32` 타입과 `Ingest Pipeline`을 사용한 과거의 복잡한 방식
* **Modern (9.x)**: `semantic_text` 필드 하나로 `bfloat16` 저장소 압축과 `BBQ` 메모리 양자화를 자동화하는 모던 방식

In [ ]:
print("🏗️ 파이프라인 및 인덱스 생성 중...")
es.indices.delete(index=[INDEX_REAL_LEGACY, INDEX_REAL_MODERN], ignore_unavailable=True)

# [Legacy] Ingest Pipeline
es.ingest.put_pipeline(id="legacy-inference-pipeline", body={"processors": [{"inference": {"model_id": "my-jina-embedding", "target_field": "title_vector", "field_map": {"title": "text_field"}}}]})

# [Legacy] float32 인덱스
es.indices.create(index=INDEX_REAL_LEGACY, body={
    "mappings": {"properties": {"title": {"type": "text"}, "title_vector": {"type": "dense_vector", "dims": VECTOR_DIMS, "element_type": "float32", "index": True}, "category": {"type": "keyword"}, "access_level": {"type": "keyword"}}}
})

# [Modern] semantic_text 인덱스
es.indices.create(index=INDEX_REAL_MODERN, body={
    "mappings": {"properties": {"content_semantic": {"type": "semantic_text", "inference_id": "my-jina-embedding"}, "title": {"type": "text", "copy_to": "content_semantic"}, "category": {"type": "keyword"}, "access_level": {"type": "keyword"}}}
})
print("✅ 인덱스 준비 완료")

## 3. 한국어 오픈 데이터(KLUE) 고속 색인 및 희소 필터(1%) 태깅
ACORN-1 알고리즘의 벤치마크를 위해 전체 문서 중 딱 **1%**에만 `access_level: premium` 태그를 부여하여 극한의 필터링 환경을 조성합니다.

In [ ]:
print("🚀 KLUE 한국어 뉴스 데이터(약 45,000건) 다운로드 및 색인 중...")
dataset = load_dataset("klue", "ynat", split="train")
label_map = {0: "IT/과학", 1: "경제", 2: "사회", 3: "생활문화", 4: "세계", 5: "스포츠", 6: "정치"}

def generate_real_data(index_name, use_pipeline=False):
    for i, row in enumerate(dataset):
        access_level = "premium" if random.random() < 0.01 else "public"
        action = {"_index": index_name, "_id": str(i), "_source": {"title": row["title"], "category": label_map.get(row["label"], "기타"), "access_level": access_level}}
        if use_pipeline: action["pipeline"] = "legacy-inference-pipeline"
        yield action

print("⏳ [Legacy] float32 임베딩 진행 중...")
deque(parallel_bulk(es, generate_real_data(INDEX_REAL_LEGACY, use_pipeline=True), thread_count=2, chunk_size=200), maxlen=0)

print("⏳ [Modern] semantic_text 자동화 색인 진행 중...")
deque(parallel_bulk(es, generate_real_data(INDEX_REAL_MODERN, use_pipeline=False), thread_count=2, chunk_size=200), maxlen=0)

print("⚙️ Lucene 세그먼트 최적화 (Force Merge) 수행 중...")
es.indices.forcemerge(index=[INDEX_REAL_LEGACY, INDEX_REAL_MODERN], max_num_segments=1)
es.indices.refresh(index=[INDEX_REAL_LEGACY, INDEX_REAL_MODERN])
print("🎉 사전 데이터 적재가 모두 완료되었습니다!")